# Data Profiling — Gaming Compliance & Risk Intelligence Platform

Profiles the **synthetic** transaction data in the Snowflake `STAGING` layer — distributions,
category mix, laundering rate, monthly volume, and data quality. Companion to the reporting
analysis notebook. **Synthetic data only; no credentials in this file.**

Run: `pip install -r ../requirements.txt`, ensure a `~/.snowflake/connections.toml` entry named
`gaming_compliance` (or `SNOWFLAKE_*` env vars), then Run All and commit with outputs.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

# No credentials in this file. Uses ~/.snowflake/connections.toml ('gaming_compliance') or SNOWFLAKE_* env vars.
try:
    conn = snowflake.connector.connect(connection_name=os.environ.get("SNOWFLAKE_CONNECTION", "gaming_compliance"))
except Exception:
    kwargs = dict(account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
                  role=os.environ.get("SNOWFLAKE_ROLE", "DATA_ENGINEER"),
                  warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", "WH_REPORTING"), database="GAMING_COMPLIANCE_DB")
    if os.environ.get("SNOWFLAKE_PASSWORD"): kwargs["password"] = os.environ["SNOWFLAKE_PASSWORD"]
    else: kwargs["authenticator"] = os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser")
    conn = snowflake.connector.connect(**kwargs)

for stmt in ("USE ROLE DATA_ENGINEER", "USE WAREHOUSE WH_REPORTING", "USE DATABASE GAMING_COMPLIANCE_DB"):
    conn.cursor().execute(stmt)

def q(sql):
    cur = conn.cursor()
    try: return cur.execute(sql).fetch_pandas_all()
    finally: cur.close()

q("SELECT CURRENT_ROLE() AS ROLE, CURRENT_WAREHOUSE() AS WH, CURRENT_DATABASE() AS DB")

## 1. Transaction overview

In [ ]:
ov = q('''SELECT COUNT(*) AS TXNS, MIN(TXN_DATE) AS FIRST_DATE, MAX(TXN_DATE) AS LAST_DATE,
                 ROUND(AVG(AMOUNT),2) AS AVG_AMT, ROUND(MEDIAN(AMOUNT),2) AS MEDIAN_AMT,
                 MIN(AMOUNT) AS MIN_AMT, MAX(AMOUNT) AS MAX_AMT
          FROM STAGING.STG_TRANSACTIONS WHERE IS_VALID''')
ov.T

## 2. Transaction amount distribution
Capped at $15k for readability (a few large values extend beyond).

In [ ]:
amt = q("SELECT AMOUNT::FLOAT AS AMOUNT FROM STAGING.STG_TRANSACTIONS WHERE IS_VALID AND AMOUNT <= 15000")
ax = amt["AMOUNT"].plot.hist(bins=40, figsize=(9,4), color="#2563eb")
ax.set_title("Transaction amount distribution (<= $15k)"); ax.set_xlabel("amount"); plt.tight_layout(); plt.show()

## 3. Payment-method mix

In [ ]:
pm = q("SELECT PAYMENT_FORMAT, COUNT(*) AS TXNS FROM STAGING.STG_TRANSACTIONS WHERE IS_VALID GROUP BY 1 ORDER BY TXNS")
ax = pm.plot.barh(x="PAYMENT_FORMAT", y="TXNS", legend=False, figsize=(8,4), color="#0ea5e9")
ax.set_title("Transactions by payment method"); ax.set_xlabel("transactions"); plt.tight_layout(); plt.show()
pm

## 4. Laundering rate by payment method
Share of transactions carrying the synthetic `IS_LAUNDERING` ground-truth label.

In [ ]:
lr = q('''SELECT PAYMENT_FORMAT, COUNT(*) AS TXNS, SUM(IFF(IS_LAUNDERING,1,0)) AS LAUNDERING,
                 ROUND(100.0*AVG(IFF(IS_LAUNDERING,1,0)),1) AS LAUNDER_PCT
          FROM STAGING.STG_TRANSACTIONS WHERE IS_VALID GROUP BY 1 ORDER BY LAUNDER_PCT DESC''')
ax = lr.plot.bar(x="PAYMENT_FORMAT", y="LAUNDER_PCT", legend=False, figsize=(9,4), color="#dc2626")
ax.set_title("Laundering rate by payment method (%)"); ax.set_ylabel("% laundering"); plt.tight_layout(); plt.show()
lr

## 5. Monthly transaction volume

In [ ]:
mv = q("SELECT TO_CHAR(TXN_DATE,'YYYY-MM') AS YM, COUNT(*) AS TXNS FROM STAGING.STG_TRANSACTIONS WHERE IS_VALID GROUP BY 1 ORDER BY 1")
ax = mv.plot(x="YM", y="TXNS", legend=False, figsize=(11,4))
ax.set_title("Monthly transaction volume"); ax.set_xlabel("month"); ax.tick_params(axis='x', labelrotation=90)
plt.tight_layout(); plt.show()

## 6. Data quality
Staging flags rows rather than dropping them; on clean synthetic data invalid rows should be 0.

In [ ]:
dq = q('''SELECT COUNT(*) AS TOTAL, SUM(IFF(IS_VALID,0,1)) AS INVALID,
                 SUM(IFF(IS_LAUNDERING,1,0)) AS LAUNDERING, SUM(IFF(SANCTIONS_FLAG,1,0)) AS SANCTIONS
          FROM STAGING.STG_TRANSACTIONS''')
dq.T

## Findings
_(Fill in after running — amount distribution shape, dominant payment methods, which methods
carry the highest laundering rate, volume trend, and DQ.)_